## PREPARING DATA

In [ ]:
#the import packages
import requests
import pandas as pd
from pandas import json_normalize
import requests
import os
from pathlib import Path
from datetime import datetime, timezone,timedelta,time
from scipy import stats
import json
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
from matplotlib.ticker import MultipleLocator


In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
def loadDataFromFile(file_name):
    script_dir = Path().resolve().parent.parent
    print(script_dir)
    data_folder = script_dir /'data'
    print(data_folder)
    data_folder.mkdir(exist_ok=True)
    
    file_path = data_folder / (file_name + ".json")
    
    if file_path.exists():
        df = pd.read_json(file_path)
        print(f"Loaded {len(df)} records from {file_path}")
        return df
    else:
        print(f"File {file_path} does not exist.")
        return None    

In [ ]:
userInputDataRaw = loadDataFromFile("UserPrevious experiments-preprocessed")
timeSeriesData_BIGraw = loadDataFromFile("Data:Previous experiments-preprocessed")

In [ ]:
timeSeriesData_BIGraw = timeSeriesData_BIGraw.set_index("seconds",drop=False)

In [ ]:
a = userInputDataRaw.index
b = timeSeriesData_BIGraw["keys"].unique()
diff_all = list(set(a).symmetric_difference(set(b)))
print(diff_all)  
userInputDataRaw.index = timeSeriesData_BIGraw["keys"].unique()
print(userInputDataRaw.index)

In [ ]:
# Convert back to timedelta
timeSeriesData_BIGraw['timestamp'] = pd.to_timedelta(timeSeriesData_BIGraw['timestamp'], unit='ms')

# Convert back to datetime

timeSeriesData_BIGraw ["datetime_timestamp"]= timeSeriesData_BIGraw['datetime_timestamp'].transform(
    lambda x: pd.to_datetime(x, unit='ms')
)


columns_datetime= [
       'date of experiment', 'actual timestamp StartingExperiment',
       'actual timestamp EndingExperiment']
columns_timedelta = ['time taken total','time taken before insertion',
       'timestamp InsertingSource timedelta',
       'time taken after insertion']
# Ensure target columns are of object type before assignment
userInputDataRaw[columns_datetime] = userInputDataRaw[columns_datetime].astype('object')
userInputDataRaw[columns_timedelta] = userInputDataRaw[columns_timedelta].astype('object')

userInputDataRaw.loc[:,columns_datetime] = userInputDataRaw.loc[:,columns_datetime].apply(lambda x:pd.to_datetime(x, unit='ms'))
userInputDataRaw.loc[:,columns_timedelta] = userInputDataRaw.loc[:,columns_timedelta].apply(lambda x:pd.to_timedelta(x, unit='ms'))

In [ ]:
timeSeriesData_BIG_Original = timeSeriesData_BIGraw.copy()
userInputData_Original = userInputDataRaw.copy()

In [ ]:
userInputData_Original

In [ ]:
userInputData_Original["room"].unique()

In [ ]:
#keep the data from the last set experiments made that have the 3 sensors in a triangle shape,they have 16 particular points in the space d
room_mask = userInputData_Original["room"].isin(['Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0'])
mask = room_mask  
userInputData_Original = userInputData_Original.loc[mask]
timeSeriesData_BIG_Original = timeSeriesData_BIG_Original.loc[timeSeriesData_BIG_Original["keys"].isin(userInputData_Original.index)]

In [ ]:
userInputData_Original

In [ ]:
timeSeriesData_BIG_Original

In [ ]:
column_to_transform = "x-y axis"
userInputData_Original.loc[:,column_to_transform] = userInputData_Original.loc[:,column_to_transform].apply(tuple)

In [ ]:
sensors_position_columns = ["position of Id="+str(id) + ":BME680:breathVocEquivalent x-y axis"
                            for id in range(3)]
sensors_euclidian_distance_columns = ["Euclidian distance to Id="+str(id) + ":BME680:breathVocEquivalent"
                                    for id in range(3)]

userInputData_Original.loc[:,sensors_position_columns] = userInputData_Original.loc[:,sensors_position_columns].applymap(lambda lst: [round(x,2) for x in lst])
userInputData_Original.loc[:,sensors_euclidian_distance_columns] = userInputData_Original.loc[:,sensors_euclidian_distance_columns].round(2)


In [ ]:
sensors_position_columns = ["position of Id="+str(id) + ":BME680:breathVocEquivalent x-y axis"
                            for id in range(3)]
column_to_transform = sensors_position_columns
userInputData_Original.loc[:,column_to_transform] = userInputData_Original.loc[:,column_to_transform].apply(tuple)

In [ ]:
userInputData_Original

In [ ]:
userInputData_Original.columns

In [ ]:

euclidian_distance_columns = ["Euclidian distance to Id="+str(id)+":BME680:breathVocEquivalent"
                               for id in range(3) ]

mask = userInputData_Original["are-doors-opened"] == "on"
for i in range(3):
    print(f"userInputData_Original[are-doors-opened] == on at id={i}:{np.sort(userInputData_Original.loc[mask,euclidian_distance_columns[i]].unique())}")
mask = userInputData_Original["are-doors-opened"] != "on"    
for i in range(3):
    print(f"userInputData_Original[are-doors-opened] != on at id={i}:{np.sort(userInputData_Original.loc[mask,euclidian_distance_columns[i]].unique())}")

In [ ]:
userInputData_Original

In [ ]:
timeSeriesData_BIG_Original

In [ ]:
userInputData_Original.loc[userInputData_Original["are-doors-opened"]=="on"].shape

In [ ]:
userInputData_Original.loc[userInputData_Original["are-doors-opened"]!="on"].shape

In [ ]:
max_before= -30
max_after = 299

column_to_check = "time taken before insertion (seconds)-capped"
mask = userInputData_Original[column_to_check] >max_before
print(f"{column_to_check} < {max_before}:\n {userInputData_Original.loc[mask,column_to_check]}")

column_to_check = "time taken after insertion (seconds)-capped"
mask = userInputData_Original[column_to_check] < max_after
print(f"{column_to_check} < {max_after}:\n {userInputData_Original.loc[mask,column_to_check]}")
print(f"userInputData_Original rows {userInputData_Original.shape[0]}")

In [ ]:
userInputData_Original.columns

## PRINT DATA PER SENSOR

### Split back into dict
dict_of_timeseries = {k: v.drop(columns="keys").reset_index(drop=True) 
             for k, v in timeSeriesData_BIG.groupby("keys")}
for index,data in dict_of_timeseries.items():
    dict_of_timeseries[index] = dict_of_timeseries[index].set_index("seconds",drop=False)

In [ ]:
def plot_position(userInputData,sample_row_of_the_group):
    plt.figure(figsize=(6, 6))  
    position_of_sensors = userInputData.iloc[-1]
    all_positions = userInputData.loc[:, ["x axis", "y axis"]]
    # Extra points
    extra_positions = np.array([
        [position_of_sensors["position of Id=0:BME680:breathVocEquivalent-x axis"], position_of_sensors["position of Id=0:BME680:breathVocEquivalent-y axis"]],
    
        [position_of_sensors["position of Id=1:BME680:breathVocEquivalent-x axis"], position_of_sensors["position of Id=1:BME680:breathVocEquivalent-y axis"]],
        [position_of_sensors["position of Id=2:BME680:breathVocEquivalent-x axis"], position_of_sensors["position of Id=2:BME680:breathVocEquivalent-y axis"]]
    ])
    extra_ids = ["id0","id1", "id2"]
    
    
    room_length = 4.0
    room_width = 3.25
    
    
    # Create scatterplot of the sources of all the particular setup
    #sns.scatterplot(x=positions[:,0], y=positions[:,1])
    sns.scatterplot(data=all_positions, x="x axis", y="y axis", color='blue', s=100, label='User Input Data')
    
    
    
    # Add the positions of sensors
    sns.scatterplot(x=extra_positions[:,0], y=extra_positions[:,1], color='red', s=100)
    x_sensor_highlight = sample_row_of_the_group["x axis"]
    y_sensor_highlight = sample_row_of_the_group["y axis"]
    # Plot a hollow circle around it
    plt.scatter(x_sensor_highlight, y_sensor_highlight, s=500, facecolors='none', edgecolors='green', linewidths=2, label='Highlighted point')  
    # Draw lines and annotate distances
    distances_from_sensors = (
        sample_row_of_the_group["Euclidian distance to Id=0:BME680:breathVocEquivalent"],
        sample_row_of_the_group["Euclidian distance to Id=1:BME680:breathVocEquivalent"],
        sample_row_of_the_group["Euclidian distance to Id=2:BME680:breathVocEquivalent"]
    )
    
    for i, (x, y) in enumerate(extra_positions):
        plt.plot([x_sensor_highlight, x], [y_sensor_highlight, y], color='red', linewidth=0.7, alpha=0.7)
        
        if distances_from_sensors is not None:
            # Midpoint of the line for annotation
            mid_x = (x_sensor_highlight + x) / 2
            mid_y = (y_sensor_highlight + y) / 2
            plt.text(mid_x, mid_y, f"{distances_from_sensors[i]:.2f}", color='red', fontsize=8, ha='center', va='center',
                         bbox=dict(facecolor='white', edgecolor='none', alpha=0.6, pad=1))
    # Annotate extra points with their IDs
    for i, (x, y) in enumerate(extra_positions):
        plt.text(x, y, extra_ids[i], fontsize=12, ha='right', va='bottom', color='red')
    
    
    # Set axis limits
    plt.xlim(-room_width, 0)
    plt.ylim(0, room_length)
    
    # Add grid
    plt.grid(True, which="both", linestyle="--", linewidth=0.7, alpha=0.7)
    # Smaller legend
    plt.legend(fontsize=8, markerscale=0.8, labelspacing=0.4)

    plt.show()

In [ ]:
def printDataBasedOnDate(column_to_print,userInputData,timeSeriesData_BIG,room_other_grouping,type_of_other_grouping):
    
    column_names_keys_color_values = {"Id=0:BME680:breathVocEquivalent":"blue","Id=1:BME680:breathVocEquivalent":"green","Id=2:BME680:breathVocEquivalent":"yellow"}
    
    for group_name,indexes_of_the_group in room_other_grouping.items(): 
        timeSeriesData_BIG_copy = timeSeriesData_BIG.copy() 

        if ("position"  in type_of_other_grouping):
            sample_row_of_the_group = userInputData.loc[indexes_of_the_group[0],:]
            
            plot_position(userInputData,sample_row_of_the_group)       
        print(f"group_name {group_name}")
        print(f"indexes_of_the_group {indexes_of_the_group}")
        data = timeSeriesData_BIG_copy.loc[timeSeriesData_BIG_copy["keys"].isin(indexes_of_the_group),:]  
      # Create relplot
        g = sns.relplot(
            data=data,
            x="seconds",
            y=column_to_print,
            hue="sensors",
            col="keys",        # separate subplot per key
            kind="line",
            col_wrap=3, 
                height=7,    # default = 5
            aspect=1, # width = height × aspect (so 6 × 1.5 = 9 inches wide per subplot
            palette=column_names_keys_color_values,  # <<< ensures the same colors across all subplots  
            linewidth=2,
           facet_kws={
            "sharex": False,
            "sharey": False       
    
        })
        

        # >>> ADD THIS PART <<<
        for ax in g.axes.flat:
            ax.xaxis.set_major_locator(MultipleLocator(30))
            ax.xaxis.set_minor_locator(MultipleLocator(10))
            ax.grid(True, which='both', linestyle=':', linewidth=0.5)
            
 
        
    # Get the horizontal and  vertical line position for this experiment
        for key_value, ax in g.axes_dict.items():
           
                #value to show the time that source is inserted
          
            userInputDataRow = userInputData.loc[key_value,:]
        #    x_position = f"side-right-wall {userInputDataRow['side-right-wall']},side-left-wall {userInputDataRow['side-left-wall']} \n"
        #    y_position = f"front-wall {userInputDataRow['front-wall']},back-wall {userInputDataRow['back-wall']} \n"
            
            euclidian_distances = (
                                  f"distance from Id0 sensor {userInputDataRow['Euclidian distance to Id=0:BME680:breathVocEquivalent']}\n",
                                  f"distance from Id1 sensor {userInputDataRow['Euclidian distance to Id=1:BME680:breathVocEquivalent']}\n",
                                  f"distance from Id2 sensor {userInputDataRow['Euclidian distance to Id=2:BME680:breathVocEquivalent']}\n",
            )
            subtitle=  (
                        f"At experiment with key {key_value}\n datetime:{userInputDataRow['actual timestamp StartingExperiment']}\n", 
                        f"experimentState:{userInputDataRow['experimentState']}\n",
                        f"x-axis: {userInputDataRow['x axis']} , y-axis: {userInputDataRow['y axis']}\n"
            )
            if ("distance"  in type_of_other_grouping):
               subtitle = "".join(subtitle) + "\n"+"".join(euclidian_distances)  
            ax.set_title(subtitle, fontsize=9)   
            g.fig.suptitle(f"Group: {group_name}", fontsize=16)
        
            g.fig.subplots_adjust(
            top=0.75,
            wspace=0.2,
            hspace=0.8   # ← increase this
        )


        plt.show()   
             

In [ ]:
def plot_all_positions(userInputData):
    room_other_grouping = userInputData.groupby(["x axis","y axis"]).groups
    
    for group_name,indexes_of_the_group in room_other_grouping.items(): 
        sample_row_of_the_group = userInputData.loc[indexes_of_the_group[0],:]
        plot_position(userInputData,sample_row_of_the_group)      
        
plot_all_positions(userInputData_Original)

In [ ]:
def plantDataGroupedByEuclidianDistance(userInputData,timeSeriesData_BIG,column_to_take):
    sensors = timeSeriesData_BIG["sensors"].unique()
    euclidian_distances_columns = [f"Euclidian distance to {sensor}" for sensor in sensors ]
    group_by_list = ["room","experimentState",*euclidian_distances_columns]
    room_other_grouping = userInputData.groupby(group_by_list).groups
    type_of_other_grouping = ["experimentState","position","distance"]
    
    sensors = timeSeriesData_BIG["sensors"].unique()
    for sensor in sensors:
        mask = timeSeriesData_BIG["sensors"] == sensor
        timeSeriesData_BIG_subset = timeSeriesData_BIG.loc[mask,:]
        print(sensor)
        printDataBasedOnDate(column_to_take,userInputData,timeSeriesData_BIG_subset,room_other_grouping,type_of_other_grouping)

In [ ]:
column_to_take = "VOC original"

plantDataGroupedByEuclidianDistance(userInputData_Original,timeSeriesData_BIG_Original,column_to_take)

In [ ]:
column_to_take = "VOC"

plantDataGroupedByEuclidianDistance(userInputData_Original,timeSeriesData_BIG_Original,column_to_take)

In [ ]:
column_to_take = "VOC rolling average"
plantDataGroupedByEuclidianDistance(userInputData_Original,timeSeriesData_BIG_Original,column_to_take)

# LIBRARIES AND FUNCTIONS DECLARACTIONS

## LIBRARIES

In [ ]:
# Required imports
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error,mean_absolute_error, r2_score

import math
from itertools import combinations
from sklearn.preprocessing import RobustScaler

## MODELS

In [ ]:

# 1. Linear Regression (no hyperparameters to tune, just for completeness)
lr = LinearRegression()
lr_params = {}  # No parameters for simple LinearRegression

# 2. Ridge Regression
ridge = Ridge()
ridge_params = {
    'alpha': [0.01, 0.1, 1, 10, 100],
    'solver': ['auto', 'svd', 'cholesky', 'lsqr']
}

# 3. Lasso Regression
lasso = Lasso()
lasso_params = {
    'max_iter':[10000],
    'alpha': [0.001, 0.01, 0.1, 1, 10],
}

# 4. ElasticNet
elastic = ElasticNet()
elastic_params = {
    'max_iter':[10000],
    'alpha': [0.001, 0.01, 0.1, 1, 10],
    'l1_ratio': [0.1, 0.5, 0.7, 0.9]
}

# 5. Support Vector Regression
svr = SVR()
svr_params = {
    'kernel':["rbf"],
    "C": [0.1, 1, 10, 100],
    "epsilon": [0.001, 0.01, 0.1, 0.2],
    "gamma": ["scale", "auto"]
}


# 7. Random Forest Regression
rf = RandomForestRegressor()
rf_params = {
    'random_state':[42],
    'n_estimators': [10, 50, 100,200],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# 8. Gradient Boosting Regression
gbr = GradientBoostingRegressor()
gbr_params = {
    'random_state':[42],
    'n_estimators': [10, 50, 100,200],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [ 5, 10, 20,None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

models = {
    'LinearRegression': (lr, lr_params),
    'Ridge': (ridge, ridge_params),
    'Lasso': (lasso, lasso_params),
    'ElasticNet': (elastic, elastic_params),
    'SVR': (svr, svr_params),
    #'RandomForest': (rf, rf_params),
    #'GradientBoosting': (gbr, gbr_params)
}

from scipy.optimize import least_squares


## FUNCTIONS

#### get_16_train_positions

In [ ]:
def get_16_train_positions(userInputData_Original:pd.DataFrame):
    column_to_check = "x-y axis"
    mask = (userInputData_Original["are-doors-opened"] != "on") & (userInputData_Original["room"] =='Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0' ) & (userInputData_Original[column_to_check] != (None,None))
    training_positions = userInputData_Original.loc[mask,column_to_check].unique()
    return training_positions


#### get_Data_To_Be_Used

In [ ]:
def get_Data_To_Be_Used(userInputData_Original:pd.DataFrame,timeSeriesData_BIG_Original:pd.DataFrame,
                        keepOpenDoorData =False,drop_other_positions =True ,closeDoorTraining_openDoorTest = False ,anyOtherMaskToBeUsed = None)->(pd.DataFrame,pd.DataFrame):

    userInputData = userInputData_Original.copy()
    timeSeriesData_BIG = timeSeriesData_BIG_Original.copy()

    # clear the data that doesn't fit to the 330 columns we want
    max_before_int= -30
    max_after_int = 299
    
    mask_before = (userInputData["time taken before insertion (seconds)-capped"] == max_before_int)
    max_after = (userInputData["time taken after insertion (seconds)-capped"] == max_after_int)
    mask = mask_before & max_after
 #   print(f"data which doesn't fit our criteria of size:{userInputData.loc[~mask,:].index}")
    userInputData = userInputData.loc[mask,:].copy()
    timeSeriesData_BIG = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)]
    if keepOpenDoorData:
        mask = (userInputData["are-doors-opened"] == "on") | (userInputData["are-doors-opened"] != "on")

    else:
        mask =  userInputData["are-doors-opened"] != "on"

    if anyOtherMaskToBeUsed is not None:
        mask = mask & anyOtherMaskToBeUsed
    userInputData = userInputData.loc[mask].copy()
    #grab all the data that are contained in those experiments
    timeSeriesData_BIG = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)].copy()
        
    
    if drop_other_positions & (not closeDoorTraining_openDoorTest):
        #from the experiments with door open, drop the positions which doesn't fit to the 16 source position we are going to check the model
        #also drop the experiments with the None values
        axis_list = get_16_train_positions(userInputData_Original)
        axis_mask = userInputData["x-y axis"].isin(axis_list)
        
        mask = axis_mask 
        userInputData = userInputData.loc[mask]
        #grab all the data that are contained in those experiments
        timeSeriesData_BIG = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)]

    return userInputData,timeSeriesData_BIG
        

#### build_X_keys_And_Y_target_Arrays

In [ ]:
def build_X_keys_And_Y_target_Arrays(userInputData,closeDoorTraining_openDoorTest = False)->(np.array,np.array):
  
    if closeDoorTraining_openDoorTest :
         pass

    else:
        keys = np.array(userInputData.index)
        keys_numpy = keys.reshape(-1,1)
   

        columns_to_pass = ["Euclidian distance to Id="+str(id)+":BME680:breathVocEquivalent"
                                   for id in range(3) ]
        column_size = len(columns_to_pass)
        target_variables = userInputData.loc[keys,columns_to_pass].to_numpy()
        target_variables_euclidian_distance_numpy = target_variables.reshape(-1,column_size)
     
    
    
        columns_to_pass = ["x-y axis"]
        target_variables = userInputData.loc[keys,columns_to_pass].to_numpy()
        target_variables_positions_numpy = np.stack(target_variables[:,0])  
        
        X_train_keys,X_test_keys,Y_train_Euclidians,Y_test_Euclidians,Y_train_Positions,Y_test_Positions = train_test_split(
            keys_numpy,target_variables_euclidian_distance_numpy,target_variables_positions_numpy,
            test_size=0.2,
            stratify=target_variables_euclidian_distance_numpy,
            random_state=42
        )
    
    return X_train_keys,X_test_keys,Y_train_Euclidians,Y_test_Euclidians,Y_train_Positions,Y_test_Positions

#### build_X_from_dataframe

In [ ]:
def build_X_from_dataframe(keys_array, df, column_to_take):
    # flatten [[159],[196]...] → [159,196,...]
    keys = keys_array.ravel()

    # number of keys
    N = len(keys)

    # number of time steps per experiment (should be constant)
    example_key = keys[0]
    time_values = df[df["keys"] == example_key]
    T = len(time_values)

    # allocate X
    X = np.zeros((N, T))

    # fill X
    for i, key in enumerate(keys):
        rows = df[df["keys"] == key]
        X[i, :] = rows[column_to_take].values

    return X

#### get_Data_Per_Sensor

In [ ]:
def get_Data_Per_Sensor(userInputData,timeSeriesData_BIG) ->pd.DataFrame:
    df_filtered = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(userInputData.index)]
    sensors_names = np.sort(timeSeriesData_BIG["sensors"].unique())
    dfs_by_sensor = {
        sensor: df_filtered.loc[df_filtered["sensors"]==sensor]
        for sensor in sensors_names
    }
    return dfs_by_sensor


In [ ]:
def build_X_train_And_X_test(userInputData,timeSeriesData_BIG,X_train_keys,X_test_keys,column_to_take ="VOC rolling average")->(np.array,np.array):
    dfs_by_sensor = get_Data_Per_Sensor(userInputData,timeSeriesData_BIG)
    X_train = np.empty((len(dfs_by_sensor),X_train_keys.shape[0],330))
    
    for i, data_per_sensor in enumerate(dfs_by_sensor.values()):
        X_train[i,:,:]= build_X_from_dataframe(X_train_keys,data_per_sensor,column_to_take)
    
    X_test = np.empty((len(dfs_by_sensor),X_test_keys.shape[0],330))
        
    for i, data_per_sensor in enumerate(dfs_by_sensor.values()):
        X_test[i,:,:]= build_X_from_dataframe(X_test_keys,data_per_sensor,column_to_take)

    return X_train,X_test

#### trimFirstColumns

In [ ]:
def trimFirstColumns(X_train,X_test,columns_to_keep = 15):
    actual_columns_to_keep = abs(-30) + columns_to_keep -1


    X_train =  X_train[:,:,actual_columns_to_keep:]
    X_test  =  X_test[:,:,actual_columns_to_keep:] 
    return X_train,X_test    

#### create_The_Arrays_For_The_Models

In [ ]:

def create_The_Arrays_For_The_Models(userInputData_Original,timeSeriesData_BIG_Original,apply_functions_to_X_data = [],
                                     column_to_take ="VOC rolling average",keepOpenDoorData =False,drop_other_positions =True ,closeDoorTraining_openDoorTest = False ,anyOtherMaskToBeUsed = None):
    
    userInputData,timeSeriesData_BIG = get_Data_To_Be_Used(userInputData_Original,timeSeriesData_BIG_Original,
                                                           keepOpenDoorData,drop_other_positions,closeDoorTraining_openDoorTest,anyOtherMaskToBeUsed)
    
    X_train_keys,X_test_keys,Y_train_Euclidians,Y_test_Euclidians,Y_train_Positions,Y_test_Positions = build_X_keys_And_Y_target_Arrays(userInputData)  
    
    X_train,X_test = build_X_train_And_X_test(userInputData,timeSeriesData_BIG,X_train_keys,X_test_keys,column_to_take)
    X_train,X_test = trimFirstColumns(X_train,X_test,columns_to_keep = 60)
    if apply_functions_to_X_data is not None and apply_functions_to_X_data is not np.nan:
       for modify_function in apply_functions_to_X_data:
           if modify_function is not None and modify_function is not np.nan:
               X_train,X_test =  modify_function(X_train,X_test)
    return X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,Y_train_Positions,Y_test_Positions, userInputData,timeSeriesData_BIG

#### findPCAcomponentsCovered

In [ ]:

    
def findPCAcomponentsCovered(X_train):     
    
    for X_train_per_sensor in X_train:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_train_per_sensor)
        # Step 2: Apply PCA
        pca = PCA()
        X_pca = pca.fit_transform(X_scaled)
        
        # Step 3: Explained variance
        explained_variance = pca.explained_variance_ratio_
        cumulative_variance = np.cumsum(explained_variance)
        
        # Only display first 10 components max
        max_components = min(10, len(explained_variance))
        ev_to_plot = explained_variance[:max_components]
        cum_to_plot = cumulative_variance[:max_components]
        
        # Step 4: Bar plot of cumulative explained variance
        plt.figure(figsize=(8,5))
        plt.bar(range(1, max_components + 1), cum_to_plot)
        plt.xlabel('Number of PCA components')
        plt.ylabel('Cumulative explained variance')
        plt.title('Cumulative explained variance (first 10 components)')
        plt.grid(True, axis='y')
        plt.show()
        
        # Step 5: Optimal number of components for ~90% variance
        optimal_components = np.argmax(cumulative_variance >= 0.90) + 1
        print("Optimal number of components to explain ~90% variance:", optimal_components)


In [ ]:
def printPCA2components(X_train,Y_train_Euclidians):
    for i in range(X_train.shape[0]):
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_train[i])
        
        # --- Step 2: Reduce to 2 components ---
        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(X_scaled)
        
        # --- Step 3: Scatter plot with hue based on Y_train ---
        plt.figure(figsize=(8,6))
        
        # Using seaborn for easy color scaling
        sns.scatterplot(
            x=X_pca[:,0], 
            y=X_pca[:,1], 
            hue=Y_train_Euclidians[:,i],              # color represents Y_train
             palette="coolwarm",         # continuous color palette
            s=80,                      # marker size
            alpha=0.8
        )
        
        plt.xlabel("PCA Component 1")
        plt.ylabel("PCA Component 2")
        plt.title("2D PCA of X_train colored by Y_train")
        plt.legend(title="Y_train", bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(True)
        plt.show()

In [ ]:

def run_grid_search_per_model(X_train, y_train,cv_number,verbose,name,model,params):
        PCA__n_components = [2,3,4,5,6,8,10]

        estimators_with_no_scaling_need = ['RandomForest','DecisionTree','GradientBoosting']

        results = {}
        if name in estimators_with_no_scaling_need:
             pipe = Pipeline([
                 ('model', model)
             ]) 
        else:
             pipe = Pipeline([
                 ('scaler', StandardScaler()),
                 ('PCA', PCA()),
                 ('model', model)
             ])
        # Build a correct param grid:
        param_grid = {
            **{'model__' + k: v for k, v in params.items()}
        }
        if name not in estimators_with_no_scaling_need:
            param_grid['PCA__n_components'] = PCA__n_components
            
        grid = GridSearchCV(
            estimator=pipe,
            param_grid=param_grid,
            cv=cv_number,
            n_jobs=-1,
            scoring='neg_mean_squared_error',
            verbose=2
        )
     
        grid.fit(X_train, y_train)
    
        results["name"]  = name
        results["model"] = grid.best_estimator_
        results["parameters"] = grid.best_params_
        results["score"] = grid.best_score_
    
        return results

def run_grid_search_find_optimal_model_per_sensor(X_train, y_train,cv_number,verbose,models):
    #put a score way worse than abest_score_nything we should have
    best_result = {
        "score" : -10,
        "model" : '',
        "parameters" : {}    
    }
    for name, (model, params) in models.items():
        print(f"Running GridSearchCV for {name}...")

        results = run_grid_search_per_model(X_train, y_train,cv_number,verbose,name,model,params)
        print(results)

        if (results["score"] >  best_result["score"]) :
            best_result["name"] = results["name"]
     #       print(best_result["name"])
            best_result["score"] = results["score"]
     #       print(best_result["score"])
            best_result["model"] = results["model"]
       #     print(best_result["model"])
            best_result["parameters"] = results["parameters"]
       #     print(best_result["parameters"])
    return best_result


def run_grid_search_find_optimal_model_per_sensor_for_all_sensors(X_train, y_train,cv_number,verbose,models):

    best_results_for_all_sensors = {}

    for i in range(X_train.shape[0]):
        best_results_for_all_sensors[i]  =run_grid_search_find_optimal_model_per_sensor(X_train[i,:,:], y_train[:,i],cv_number,verbose,models)

    return best_results_for_all_sensors

In [ ]:
#PREDICT AND SEE INDIVIDUAL RESULTS
def getPredictedValues(X_test,Y_test_Euclidians,best_results_for_all_sensors):

   predict_Euclidians = np.empty((Y_test_Euclidians.shape))
    
    
   for i in range(X_test.shape[0]):
        print(f"For sensor with id={i}")
        predict_Euclidians[:,i] = best_results_for_all_sensors[i]["model"].predict(X_test[i,:,:])
    
   return predict_Euclidians
       

In [ ]:
def trainModels_PrintResults_AndGetPredictedValues(userInputData, X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,models):
    cv_number = 5
    verbose = 0
    
    best_results_for_all_sensors = run_grid_search_find_optimal_model_per_sensor_for_all_sensors(X_train, Y_train_Euclidians,cv_number,verbose,models)
    
    Y_predict_Euclidians = getPredictedValues(X_test,Y_test_Euclidians,best_results_for_all_sensors)
    plantAllBaseSensorsData(Y_test_Euclidians, Y_predict_Euclidians,"Euclidian distance",userInputData)
    return Y_predict_Euclidians

In [ ]:
def grabPositionOfSensors(userInputData):
 # --- SENSOR POINTS (red true, blue/orange predicted) ---
    sensors_position_columns = ["position of Id="+str(id) + ":BME680:breathVocEquivalent x-y axis"
                            for id in range(3)]
    point_of_sensors = []
    mask = userInputData["room"] == 'Κρεβατοκάμαρα id:0 Μ:0.5 Α:1.4 ,id:1 Π:1.7 Δ:0.5  ,id:2 Π:0.4 Α:1.0'
    for sensors_position_column in sensors_position_columns:
        
        point_of_sensors.append(userInputData.loc[mask,sensors_position_column].iloc[0]) 

    return   point_of_sensors 

#### plot_pred_vs_actual

In [ ]:
def plot_pred_vs_actual(y_test, y_pred, sensor_name, AXIS,userInputData):

    mse = mean_squared_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)

    # convert to 2D format
    if y_test.ndim == 1:
        y_test = y_test.reshape(-1,1)
    if y_pred.ndim == 1:
        y_pred = y_pred.reshape(-1,1)

    # Euclidean error
    EucDis = np.linalg.norm(y_test - y_pred, axis=1).mean()
    
    print(f"\n--- SENSOR {sensor_name} ---")
    print(f"MSE on test set: {mse:.4f}")
    print(f"R^2 on test set: {r2:.4f}")
    print(f"Euclidean distance mean: {EucDis:.4f}")

    # If 2D case
    if y_test.shape[1] == 2:

        plt.figure(figsize=(8,7))

        # --- TRUE (blue) ---
        plt.scatter(y_test[:,0], y_test[:,1], label="True", alpha=0.7, color="blue")
        
        # Print ID next to each true point
        for idx, (x,y) in enumerate(y_test):
            plt.text(x, y, str(idx), color="blue", fontsize=9)

        # --- PREDICTED (orange) ---
        plt.scatter(y_pred[:,0], y_pred[:,1], label="Predicted", alpha=0.7, color="orange")
        
        # Print ID next to each predicted point
        for idx, (x,y) in enumerate(y_pred):
            plt.text(x, y, str(idx), color="orange", fontsize=9)

        point_of_sensors = grabPositionOfSensors(userInputData)
        if point_of_sensors is not None:
            point_of_sensors = np.array(point_of_sensors)

            # TRUE sensor points → BLUE  
            plt.scatter(point_of_sensors[:,0], point_of_sensors[:,1],
                        marker="X", s=200, color="red", label="Sensor Base Points")

        plt.legend()
        plt.xlabel("X")
        plt.ylabel("Y")
        plt.title(f"Sensor {sensor_name} - True vs Predicted")
        plt.grid(True)
        plt.show()

    else:
        # fallback for 1D case
        plt.figure(figsize=(7,6))
        plt.scatter(y_test, y_pred, s=40)
        plt.plot([y_test.min(), y_test.max()], 
                 [y_test.min(), y_test.max()], linestyle='--')
        plt.xlabel("True")
        plt.ylabel("Predicted")
        plt.title(f"{sensor_name} (1D) predictions vs actual")
        plt.grid(True)
        plt.show()
    return -mse, -EucDis, r2   
def plantAllBaseSensorsData(y_test, y_pred,AXIS,userInputData):
    for i in range(y_test.shape[1]):
        AXIS = "Euclidian distances"
        plot_pred_vs_actual(y_test[:,i], y_pred[:,i],i,AXIS,userInputData)


In [ ]:
def trilateration_least_squares(points, radii, tol=5e-2):
    def fun(p):
        x, y = p
        return [np.hypot(x - px, y - py) - r for (px, py), r in zip(points, radii)]

    x0 = [np.mean([px for px,_ in points]),
          np.mean([py for _,py in points])]

    res = least_squares(fun, x0=x0)
    x, y = res.x

    # Residuals close to zero?
    boundary_ok = np.all(np.abs(fun([x, y])) < tol)

    # Check if point lies inside all circles
    inside_ok = all(
        np.hypot(x - px, y - py) <= r + tol
        for (px, py), r in zip(points, radii)
    )

    has_common_point = inside_ok  # this is the correct criterion

    return has_common_point, (x, y)
 

#### findIntersectionOfCircles

In [ ]:
#Take the three points of the sensors
def findIntersectionOfCircles(predict_Euclidians,Y_test_Positions,userInputData):
    
    point_of_sensors = grabPositionOfSensors(userInputData)

    predict_Positions = np.empty((predict_Euclidians.shape[0],2))
    threshold_values =  np.empty((predict_Euclidians.shape[0],1))
    for i in range(predict_Euclidians.shape[0]):
        threshold,point = trilateration_least_squares(point_of_sensors,predict_Euclidians[i,:])
        predict_Positions[i,:] = point
        threshold_values[i] = threshold
    
    MSE, EUCLIDIAN, R2 = plot_pred_vs_actual(Y_test_Positions,predict_Positions,"Final","Point predicted",userInputData)
    return predict_Positions,threshold_values,MSE, EUCLIDIAN, R2

#### PIPELINE

In [ ]:

def runPipeline(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,extra_params ={},apply_functions_to_X_data = []):
    print("aaaaaaaaaa")
    print(extra_params)
    (X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,Y_train_Positions,Y_test_Positions, userInputData,
        timeSeriesData_BIG) = create_The_Arrays_For_The_Models(userInputData_Original,timeSeriesData_BIG_Original,apply_functions_to_X_data,column_to_take,**extra_params)
    print(Y_test_Euclidians.shape)
    printPCA2components(X_train,Y_train_Euclidians)
    predict_Euclidians = trainModels_PrintResults_AndGetPredictedValues(userInputData, X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,models)

    predict_Positions,threshold_values,MSE, EUCLIDIAN, R2 = findIntersectionOfCircles(predict_Euclidians,Y_test_Positions,userInputData)

    return (predict_Positions,predict_Euclidians,X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,
    Y_train_Positions,Y_test_Positions, userInputData,timeSeriesData_BIG,predict_Euclidians,predict_Positions,threshold_values,
           MSE,EUCLIDIAN,R2)

In [ ]:
def circle_intersections(c1, r1, c2, r2):
    x0, y0 = c1
    x1, y1 = c2

    dx = x1 - x0
    dy = y1 - y0
    d = math.hypot(dx, dy)

    # No intersection
    if d > r1 + r2 or d < abs(r1 - r2) or d == 0:
        return []

    # Compute a, h
    a = (r1**2 - r2**2 + d**2) / (2*d)
    h = math.sqrt(max(0, r1**2 - a**2))

    xm = x0 + a * dx / d
    ym = y0 + a * dy / d

    xs1 = xm + h * dy / d
    ys1 = ym - h * dx / d
    xs2 = xm - h * dy / d
    ys2 = ym + h * dx / d

    return [(xs1, ys1), (xs2, ys2)]

# ------------------------------------------------------------
# Collect intersection points that lie inside all circles
# ------------------------------------------------------------
def intersection_points_of_all_circles(points, radii, tol=1e-6):
    candidates = []

    # pairwise intersections
    for (p1, r1), (p2, r2) in combinations(zip(points, radii), 2):
        pts = circle_intersections(p1, r1, p2, r2)
        candidates.extend(pts)

    # keep only points that lie inside ALL circles
    inside = []
    for x, y in candidates:
        if all(math.hypot(x - px, y - py) <= r + tol
               for (px, py), r in zip(points, radii)):
            inside.append((x, y))

    return inside

# ------------------------------------------------------------
# Plotting function
# ------------------------------------------------------------
def plot_circles_and_intersections(points, radii,i,at):
    fig, ax = plt.subplots(figsize=(6, 6))

    # Draw all circles
    for (px, py), r in zip(points, radii):
        circle = plt.Circle((px, py), r, fill=False, lw=2)
        ax.add_patch(circle)
        ax.plot(px, py, 'ko')  # centers

    # Compute intersections
    ints = intersection_points_of_all_circles(points, radii)

    # Plot intersection points
    for x, y in ints:
        ax.plot(x, y, 'ro', markersize=8)

    ax.set_aspect('equal')
    ax.grid(True)
    ax.set_title("Circles and Intersection Points for row " + str(i) + " at "+at)
    plt.show()

## FILTER FUNCTIONS

In [ ]:
def cutOutliers(X_train:np.array,X_test:np.array)->(np.array,np.array):

    for i in range(X_train.shape[0]):
  
        q25 = np.quantile(X_train[i,:,:], 0.25)
        q75 = np.quantile(X_train[i,:,:], 0.75)
    
        # Limit (clip) all values to the range [25% quantile, 75% quantile] for both train and test,while have learned the quantiles of train
        X_train[i,:,:] = np.clip(X_train[i,:,:], q25, q75)
        X_test[i,:,:] = np.clip(X_test[i,:,:], q25, q75)

    return X_train,X_test    

In [ ]:
def cutOutliersPerColumn(X_train:np.array,X_test:np.array)->(np.array,np.array):

    for i in range(X_train.shape[0]):
        for j in range(X_train.shape[2]):
            q25 = np.quantile(X_train[i,:,j], 0.25)
            q75 = np.quantile(X_train[i,:,j], 0.75)
        
            # Limit (clip) all values to the range [25% quantile, 75% quantile] for both train and test,while have learned the quantiles of train
            X_train[i,:,j] = np.clip(X_train[i,:,j], q25, q75)
            X_test[i,:,j] = np.clip(X_test[i,:,j], q25, q75)        

    return X_train,X_test       

In [ ]:
def cutOutliersUpper(X_train:np.array,X_test:np.array)->(np.array,np.array):

    for i in range(X_train.shape[0]):
  
        q25 = np.quantile(X_train[i,:,:], 0.25)
        q75 = np.quantile(X_train[i,:,:], 0.75)
    
        # Limit (clip) all values to the range [25% quantile, 75% quantile] for both train and test,while have learned the quantiles of train
        X_train[i,:,:] = np.clip(X_train[i,:,:], None, q75)
        X_test[i,:,:] = np.clip(X_test[i,:,:], None, q75)

    return X_train,X_test    

In [ ]:
def cutOutliersUpperPerColumn(X_train:np.array,X_test:np.array)->(np.array,np.array):

    for i in range(X_train.shape[0]):
        for j in range(X_train.shape[2]):
            q25 = np.quantile(X_train[i,:,j], 0.25)
            q75 = np.quantile(X_train[i,:,j], 0.75)
        
            # Limit (clip) all values to the range [25% quantile, 75% quantile] for both train and test,while have learned the quantiles of train
            X_train[i,:,j] = np.clip(X_train[i,:,j], None, q75)
            X_test[i,:,j] = np.clip(X_test[i,:,j], None, q75)        

    return X_train,X_test       

In [ ]:
def createGradient(X_train:np.array,X_test:np.array)->(np.array,np.array):

    for i in range(X_train.shape[0]):
        for j in range(X_train.shape[1]):
           
            # Limit (clip) all values to the range [25% quantile, 75% quantile] for both train and test,while have learned the quantiles of train
            X_train[i,j,:] = np.gradient(X_train[i,j,:])
            
    for i in range(X_test.shape[0]):
        for j in range(X_test.shape[1]):
           
            # Limit (clip) all values to the range [25% quantile, 75% quantile] for both train and test,while have learned the quantiles of train
            X_test[i,j,:] = np.gradient(X_test[i,j,:])        

    return X_train,X_test       

# FIND THE VARIABLES 

## first type

In [ ]:
column_to_take ="VOC rolling average"
apply_functions_to_X_data = [cutOutliers]

(predict_Positions,predict_Euclidians,X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,
    Y_train_Positions,Y_test_Positions, userInputData,timeSeriesData_BIG,predict_Euclidians,points,threshold_values) = runPipeline(userInputData_Original,timeSeriesData_BIG_Original,models)


In [ ]:
points

In [ ]:
predict_Euclidians

In [ ]:
Y_test_Positions

In [ ]:
Y_test_Euclidians

In [ ]:
findIntersectionOfCircles(Y_test_Euclidians,Y_test_Positions,userInputData)

In [ ]:
point_of_sensors = grabPositionOfSensors(userInputData)

for i in range(Y_test_Positions.shape[0]):
    plot_circles_and_intersections(point_of_sensors, Y_test_Euclidians[i,:],i,"Y_test_Euclidians")

In [ ]:
point_of_sensors = grabPositionOfSensors(userInputData)

for i in range(predict_Euclidians.shape[0]):
    plot_circles_and_intersections(point_of_sensors, predict_Euclidians[i,:],i,"predict_Euclidians")

In [ ]:

findPCAcomponentsCovered(X_train)


for i in range(X_train.shape[0]):
    scaler = StandardScaler()
    cutOutliers()
    X_scaled = scaler.fit_transform(X_train[i])
    
    # --- Step 2: Reduce to 2 components ---
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    
    # --- Step 3: Scatter plot with hue based on Y_train ---
    plt.figure(figsize=(8,6))
    
    # Using seaborn for easy color scaling
    sns.scatterplot(
        x=X_pca[:,0], 
        y=X_pca[:,1], 
        hue=Y_train_Euclidians[:,i],              # color represents Y_train
        palette="viridis",         # continuous color palette
        s=80,                      # marker size
        alpha=0.8
    )
    
    plt.xlabel("PCA Component 1")
    plt.ylabel("PCA Component 2")
    plt.title("2D PCA of X_train colored by Y_train")
    plt.legend(title="Y_train", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True)
    plt.show()

In [ ]:
X_train_temp , X_test_temp = cutOutliers(X_train,X_test)

for i in range(X_train.shape[0]):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train[i])
    
    # --- Step 2: Reduce to 2 components ---
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    
    # --- Step 3: Scatter plot with hue based on Y_train ---
    plt.figure(figsize=(8,6))
    
    # Using seaborn for easy color scaling
    sns.scatterplot(
        x=X_pca[:,0], 
        y=X_pca[:,1], 
        hue=Y_train_Euclidians[:,i],              # color represents Y_train
        palette="viridis",         # continuous color palette
        s=80,                      # marker size
        alpha=0.8
    )
    
    plt.xlabel("PCA Component 1")
    plt.ylabel("PCA Component 2")
    plt.title("2D PCA of X_train colored by Y_train")
    plt.legend(title="Y_train", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True)
    plt.show()

In [ ]:
timeSeriesData_BIG

## CREATE DATAFRAMES AND FUNCTION TO PRINT

In [ ]:
functions_to_apply_Outliers = [None,cutOutliers,cutOutliersPerColumn,cutOutliersUpper,cutOutliersUpperPerColumn]
functions_to_apply_Gradient = [None,createGradient]
functions_to_apply_Outliers_names = [X.__name__ if X is not None else X for X in functions_to_apply_Outliers]
functions_to_apply_Gradient_names = [X.__name__ if X is not None else X for X in functions_to_apply_Gradient]

column_to_take =["VOC original","VOC","VOC rolling average"]
room_cases = ["Only closed","All data"]
multi_index = pd.MultiIndex.from_product([room_cases,column_to_take,functions_to_apply_Gradient_names,functions_to_apply_Outliers_names],names= ["room cases" ,"column to take","Gradient functions","Outlier functions"])
for idx in multi_index:
    print(idx)

In [ ]:
columns = ["MSE","EUCLIDIAN","R2"]
parametersResultsGathered = pd.DataFrame(data=None,index=multi_index,columns=columns)
parametersResultsGathered

In [ ]:
def loopTroughFunctions(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,room_case,extra_params,parametersResultsGathered):
    print(column_to_take)
    print(extra_params)
    functions_to_apply_Outliers = [None,cutOutliers,cutOutliersPerColumn,cutOutliersUpper,cutOutliersUpperPerColumn]
    functions_to_apply_Gradient = [None,createGradient]

    
    for function_Gradient in functions_to_apply_Gradient:

        for function_Outlier in functions_to_apply_Outliers:
            outlier_name = function_Outlier.__name__ if function_Outlier is not None else np.nan
            gradient_name = function_Gradient.__name__ if function_Gradient is not None else np.nan
            print("outlier_name "+str(outlier_name))
            print("gradient_name "+str(gradient_name))


            apply_functions_to_X_data = [function_Outlier,function_Gradient]

            (predict_Positions,predict_Euclidians,X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,
                Y_train_Positions,Y_test_Positions, userInputData,timeSeriesData_BIG,
                 predict_Euclidians,points,threshold_values,
                 MSE,EUCLIDIAN,R2 ) = runPipeline(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,extra_params,apply_functions_to_X_data)

            idx = (room_case, column_to_take, gradient_name, outlier_name)
            parametersResultsGathered.loc[idx, ["MSE", "EUCLIDIAN", "R2"]] = [MSE, EUCLIDIAN, R2]
            print(parametersResultsGathered.loc[idx, ["MSE", "EUCLIDIAN", "R2"]])
    return parametersResultsGathered      
            # MSE,EUCLIDIAN,R2 the values i want to insert 

## RUN THE FUNCTIONS

### ONLY CLOSED

#### VOC Original

In [ ]:
column_to_take = "VOC original"
room_case = "Only closed"
extra_params = { "keepOpenDoorData" :False}

parametersResultsGathered = loopTroughFunctions(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,room_case,extra_params,parametersResultsGathered)

In [ ]:
parametersResultsGathered.loc[room_case]

#### VOC

In [ ]:
column_to_take = "VOC"
room_case = "Only closed"
extra_params = { "keepOpenDoorData" :False}

parametersResultsGathered = loopTroughFunctions(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,room_case,extra_params,parametersResultsGathered)

In [ ]:
parametersResultsGathered.loc[room_case]

#### VOC rolling average

In [ ]:
column_to_take = "VOC rolling average"
room_case = "Only closed"
extra_params = { "keepOpenDoorData" :False}

parametersResultsGathered = loopTroughFunctions(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,room_case,extra_params,parametersResultsGathered)

In [ ]:
parametersResultsGathered.loc[room_case]

In [ ]:
df_sorted = parametersResultsGathered.sort_values(
    by="EUCLIDIAN",
    key=lambda col: col.abs()
)
df_sorted

### ALL DATA

#### VOC rolling average

#### VOC Original

In [ ]:
column_to_take = "VOC original"
room_case = "All data"
extra_params = { "keepOpenDoorData" :True}

parametersResultsGathered = loopTroughFunctions(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,room_case,extra_params,parametersResultsGathered)

In [ ]:
parametersResultsGathered.loc[room_case]

#### VOC

In [ ]:
column_to_take = "VOC"
room_case = "All data"
extra_params = { "keepOpenDoorData" :True}

parametersResultsGathered = loopTroughFunctions(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,room_case,extra_params,parametersResultsGathered)

In [ ]:
parametersResultsGathered.loc[room_case]

In [ ]:
column_to_take = "VOC rolling average"
room_case = "All data"
extra_params = { "keepOpenDoorData" :True}

parametersResultsGathered = loopTroughFunctions(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,room_case,extra_params,parametersResultsGathered)

In [ ]:
parametersResultsGathered.loc[room_case]

In [ ]:
parametersResultsGathered.head(60)

In [ ]:
df_sorted = parametersResultsGathered.sort_values(
    by="EUCLIDIAN",
    key=lambda col: col.abs()
)
df_sorted

In [ ]:
column_to_take = "VOC original"
room_case = "Only closed"
extra_params = { "keepOpenDoorData" :False}

In [ ]:
column_to_take = "VOC original"
extra_params = { "keepOpenDoorData" :False}
apply_functions_to_X_data = [createGradient,cutOutliersUpperPerColumn]

(predict_Positions,predict_Euclidians,X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,
Y_train_Positions,Y_test_Positions, userInputData,timeSeriesData_BIG,
 predict_Euclidians,points,threshold_values,
 MSE,EUCLIDIAN,R2 ) = runPipeline(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,extra_params,apply_functions_to_X_data)

In [ ]:
Y_test_Positions.shape

In [ ]:
column_to_take = "VOC original"
extra_params = { "keepOpenDoorData" :True}
apply_functions_to_X_data = [createGradient,cutOutliersUpper]

(predict_Positions,predict_Euclidians,X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,
Y_train_Positions,Y_test_Positions, userInputData,timeSeriesData_BIG,
 predict_Euclidians,points,threshold_values,
 MSE,EUCLIDIAN,R2 ) = runPipeline(userInputData_Original,timeSeriesData_BIG_Original,models,column_to_take,extra_params,apply_functions_to_X_data)

In [ ]:
Y_test_Positions.shape

## END 

# FEATURE SEARCH 

## VOC ORIGINAL 

### NO TRIMMING

In [ ]:

X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,Y_train_Positions,Y_test_Positions, userInputData,timeSeriesData_BIG = create_The_Arrays_For_The_Models(userInputData_Original,timeSeriesData_BIG_Original,apply_functions_to_X_data = [],
                                     column_to_take ="VOC original",keepOpenDoorData =False,drop_other_positions =True ,closeDoorTraining_openDoorTest = False ,anyOtherMaskToBeUsed = None)

In [ ]:
X_whole_true = np.empty((X_train.shape[0],X_train.shape[1]+X_test.shape[1],X_train.shape[2]))
for i,_ in enumerate(X_whole_true):
    X_whole_true[i,:,:] = np.vstack((X_train[i,:,:],X_test[i,:,:]))
print(X_whole_true.shape)
X_whole_true

In [ ]:
for data in X_whole_true:
    negative_values = data[data < 0] 
    print(negative_values)

In [ ]:
min_value = {}
for i,data in enumerate(X_whole_true):
    min_value[i] =data[data > 0].min()
    print(min_value[i])

In [ ]:
min_value

In [ ]:
X_whole = np.empty((X_whole_true.shape))
for i,data in enumerate(X_whole_true):


    X_whole[i,:,:]  = np.where(data > 0, data, min_value[i])


In [ ]:
for data in X_whole:
    negative_values = data[data < 0] 
    print(negative_values)

In [ ]:
for i,data in enumerate(X_whole):

    data_flatten = data.ravel()
    
    plt.figure(figsize=(20, 3))  # ultra-wide & flat
    sns.boxplot(x=data_flatten)
    
    # Custom ticks for the first part of the plot
    ticks = np.arange(1, 3)  # 1,2,...,10
    #plt.xticks(ticks, labels=ticks)
    plt.title(f"Box plot for sensor with id={i}")

    plt.show()

In [ ]:
for i,data in enumerate(X_whole):

    X_whole_flatten = data.ravel()
    
    plt.figure(figsize=(20, 3))  # ultra-wide & flat
    plt.title(f"Box plot with no outliers for sensor with id={i}")

    sns.boxplot(x=X_whole_flatten, showfliers=False)
    plt.show()

In [ ]:
for i,data in enumerate(X_whole):
    
    X_whole_flatten = data.ravel()
    
    plt.figure(figsize=(12, 4))
    sns.boxplot(x=X_whole_flatten)
    plt.xscale("log")
    # Custom ticks for the first part of the plot
    ticks = np.arange(1, 10)  # 1,2,...,10
    plt.xticks(ticks, labels=ticks)
    plt.title(f"Box plot Log scaled for sensor with id={i}")
    plt.show()

In [ ]:
for sensor_id, data in enumerate(X_whole):
    X_whole_flatten = data.ravel()
    X_whole_flatten = np.sort(X_whole_flatten)
    
    n_plots = 4
    chunks = np.array_split(X_whole_flatten, n_plots)
    
    fig, axes = plt.subplots(n_plots, 1, figsize=(14, 10))
    
    # Figure-wide title
    fig.suptitle(f"Hist plots for sensor with id={sensor_id}", fontsize=16, y=1.02)
    
    for j, (chunk, ax) in enumerate(zip(chunks, axes)):
        sns.histplot(
            chunk,
            bins=50,
            ax=ax
        )
        ax.set_title(f"Histogram {j+1}")
    
    # Adjust layout to prevent overlap
    plt.tight_layout()
    plt.show()

#### PLOT TIME SERIES

In [ ]:
def plantDataGroupedByEuclidianDistance(userInputData,timeSeriesData_BIG,column_to_take):
    sensors = timeSeriesData_BIG["sensors"].unique()
    euclidian_distances_columns = [f"Euclidian distance to {sensor}" for sensor in sensors ]
    group_by_list = ["room","experimentState",*euclidian_distances_columns]
    room_other_grouping = userInputData.groupby(group_by_list).groups
    type_of_other_grouping = ["experimentState","position","distance"]
    
    sensors = timeSeriesData_BIG["sensors"].unique()
    for sensor in sensors:
        mask = timeSeriesData_BIG["sensors"] == sensor
        timeSeriesData_BIG_subset = timeSeriesData_BIG.loc[mask,:]
        print(sensor)
        printDataBasedOnDate(column_to_take,userInputData,timeSeriesData_BIG_subset,room_other_grouping,type_of_other_grouping)

In [ ]:
column_to_take = "VOC original"

plantDataGroupedByEuclidianDistance(userInputData,timeSeriesData_BIG,column_to_take)

### TRIM UNTIL THE 60 SECONDS AFTER 

In [ ]:
column_to_take = "VOC"
X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,Y_train_Positions,Y_test_Positions, userInputData,timeSeriesData_BIG = create_The_Arrays_For_The_Models(userInputData_Original,timeSeriesData_BIG_Original,apply_functions_to_X_data = [],
                                     column_to_take =column_to_take,keepOpenDoorData =False,drop_other_positions =True ,closeDoorTraining_openDoorTest = False ,anyOtherMaskToBeUsed = None)


columns_to_trim = abs(-30) + 60 -1
X_whole_true = np.empty((X_train.shape[0],X_train.shape[1]+X_test.shape[1],(X_train.shape[2] - columns_to_trim )))
for i,_ in enumerate(X_whole):
    data = np.vstack((X_train[i,:,:],X_test[i,:,:]))
    X_whole_true[i,:,:] = data[:,columns_to_trim:]
print(X_whole_true.shape)
X_whole_true

In [ ]:
for data in X_whole:
    negative_values = data[data < 0] 
    print(negative_values)

#### DATA DISTRIBUTION PLOT 

In [ ]:
for i,data in enumerate(X_whole):

    data_flatten = data.ravel()
    
    plt.figure(figsize=(20, 3))  # ultra-wide & flat
    sns.boxplot(x=data_flatten)
    
    # Custom ticks for the first part of the plot
    ticks = np.arange(1, 3)  # 1,2,...,10
    #plt.xticks(ticks, labels=ticks)
    #plt.title(f"Box plot for sensor with id={i}")

    plt.show()

In [ ]:
for i,data in enumerate(X_whole):

    X_whole_flatten = data.ravel()
    
    plt.figure(figsize=(20, 3))  # ultra-wide & flat
    plt.title(f"Box plot with no outliers for sensor with id={i}")

    sns.boxplot(x=X_whole_flatten, showfliers=False)
    plt.show()

In [ ]:
for i,data in enumerate(X_whole):
    
    X_whole_flatten = data.ravel()
    
    plt.figure(figsize=(12, 4))
    sns.boxplot(x=X_whole_flatten)
    plt.xscale("log")
    # Custom ticks for the first part of the plot
    ticks = np.arange(1, 10)  # 1,2,...,10
    plt.xticks(ticks, labels=ticks)
    plt.title(f"Box plot Log scaled for sensor with id={i}")
    plt.show()

In [ ]:
for sensor_id, data in enumerate(X_whole):
    X_whole_flatten = data.ravel()
    X_whole_flatten = np.sort(X_whole_flatten)
    
    n_plots = 4
    chunks = np.array_split(X_whole_flatten, n_plots)
    
    fig, axes = plt.subplots(n_plots, 1, figsize=(14, 10))
    
    # Figure-wide title
    fig.suptitle(f"Hist plots for sensor with id={sensor_id}", fontsize=16, y=1.02)
    
    for j, (chunk, ax) in enumerate(zip(chunks, axes)):
        sns.histplot(
            chunk,
            bins=50,
            ax=ax
        )
        ax.set_title(f"Histogram {j+1}")
    
    # Adjust layout to prevent overlap
    plt.tight_layout()
    plt.show()

#### PLOT TIME SERIES

In [ ]:
def plantDataGroupedByEuclidianDistance(userInputData,timeSeriesData_BIG,column_to_take):
    sensors = timeSeriesData_BIG["sensors"].unique()
    euclidian_distances_columns = [f"Euclidian distance to {sensor}" for sensor in sensors ]
    group_by_list = ["room","experimentState",*euclidian_distances_columns]
    room_other_grouping = userInputData.groupby(group_by_list).groups
    type_of_other_grouping = ["experimentState","position","distance"]
    
    sensors = timeSeriesData_BIG["sensors"].unique()
    for sensor in sensors:
        mask = timeSeriesData_BIG["sensors"] == sensor
        timeSeriesData_BIG_subset = timeSeriesData_BIG.loc[mask,:]
        print(sensor)
        printDataBasedOnDate(column_to_take,userInputData,timeSeriesData_BIG_subset,room_other_grouping,type_of_other_grouping)

In [ ]:
timeSeriesData_BIG

In [ ]:
column_to_take = "VOC"
timeSeriesData_BIG_temp = timeSeriesData_BIG.loc[timeSeriesData_BIG["seconds"]>=60]
timeSeriesData_BIG_temp

In [ ]:
plantDataGroupedByEuclidianDistance(userInputData,timeSeriesData_BIG_temp,column_to_take)

#### MOST EXPLORING

In [ ]:
groups ={}
for i in range(X_train.shape[0]):
    groups[i] = {
        val: X_train[i,:,:][Y_train_Euclidians[:,i] == val]
        for val in np.unique(Y_train_Euclidians[:,i])
    }
    
groups

In [ ]:
def heatmapConfusionMatrix(sensor,target_variable,rows_of_data):
    # Correlation matrix
    print("rows_of_data shape:", rows_of_data.shape)
    corr_matrix = np.corrcoef(rows_of_data)
    # Heatmap
    print(corr_matrix)
    plt.figure(figsize=(13, 8))  # square & larger

    sns.heatmap(
        corr_matrix,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        vmin=-1,
        vmax=1,
        square=True,
        annot_kws={"size": 10},   # 👈 key fix
        cbar_kws={"label": "Correlation"}
    )
    
    plt.title(f"Correlation Matrix Heatmap for sensor with id={sensor},at euclidian distance {target_variable}")
    plt.show()

In [ ]:
for sensor,groups_of_data_based_on_target_variable in groups.items():
    for target_variable,rows_of_data in groups_of_data_based_on_target_variable.items():
        print(rows_of_data.shape)
        heatmapConfusionMatrix(sensor,target_variable,rows_of_data)

## VOC  

### NO TRIMMING

In [ ]:
column_to_take = "VOC"
X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,Y_train_Positions,Y_test_Positions, userInputData,timeSeriesData_BIG = create_The_Arrays_For_The_Models(userInputData_Original,timeSeriesData_BIG_Original,apply_functions_to_X_data = [],
                                     column_to_take =column_to_take,keepOpenDoorData =False,drop_other_positions =True ,closeDoorTraining_openDoorTest = False ,anyOtherMaskToBeUsed = None)

In [ ]:
X_whole_true = np.empty((X_train.shape[0],X_train.shape[1]+X_test.shape[1],X_train.shape[2]))
for i,_ in enumerate(X_whole_true):
    X_whole_true[i,:,:] = np.vstack((X_train[i,:,:],X_test[i,:,:]))
print(X_whole_true.shape)
X_whole_true

In [ ]:
for data in X_whole_true:
    negative_values = data[data < 0] 
    print(negative_values)

In [ ]:
min_value = {}
for i,data in enumerate(X_whole_true):
    min_value[i] =data[data > 0].min()
    print(min_value[i])

In [ ]:
min_value

In [ ]:
X_whole = np.empty((X_whole_true.shape))
for i,data in enumerate(X_whole_true):


    X_whole[i,:,:]  = np.where(data > 0, data, min_value[i])


In [ ]:
for data in X_whole:
    negative_values = data[data < 0] 
    print(negative_values)

In [ ]:
for i,data in enumerate(X_whole):

    data_flatten = data.ravel()
    
    plt.figure(figsize=(20, 3))  # ultra-wide & flat
    sns.boxplot(x=data_flatten)
    
    # Custom ticks for the first part of the plot
    ticks = np.arange(1, 3)  # 1,2,...,10
    #plt.xticks(ticks, labels=ticks)
    plt.title(f"Box plot for sensor with id={i}")

    plt.show()

In [ ]:
for i,data in enumerate(X_whole):

    X_whole_flatten = data.ravel()
    
    plt.figure(figsize=(20, 3))  # ultra-wide & flat
    plt.title(f"Box plot with no outliers for sensor with id={i}")

    sns.boxplot(x=X_whole_flatten, showfliers=False)
    plt.show()

In [ ]:
for i,data in enumerate(X_whole):
    
    X_whole_flatten = data.ravel()
    
    plt.figure(figsize=(12, 4))
    sns.boxplot(x=X_whole_flatten)
    plt.xscale("log")
    # Custom ticks for the first part of the plot
    #ticks = np.arange(1, 10)  # 1,2,...,10
    #plt.xticks(ticks, labels=ticks)
    plt.title(f"Box plot Log scaled for sensor with id={i}")
    plt.show()

In [ ]:
for sensor_id, data in enumerate(X_whole):
    X_whole_flatten = data.ravel()
    X_whole_flatten = np.sort(X_whole_flatten)
    
    n_plots = 4
    chunks = np.array_split(X_whole_flatten, n_plots)
    
    fig, axes = plt.subplots(n_plots, 1, figsize=(14, 10))
    
    # Figure-wide title
    fig.suptitle(f"Hist plots for sensor with id={sensor_id}", fontsize=16, y=1.02)
    
    for j, (chunk, ax) in enumerate(zip(chunks, axes)):
        sns.histplot(
            chunk,
            bins=50,
            ax=ax
        )
        ax.set_title(f"Histogram {j+1}")
    
    # Adjust layout to prevent overlap
    plt.tight_layout()
    plt.show()

### TRIM FIRST 45 COLUMNS

In [ ]:
column_to_take = "VOC"
X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,Y_train_Positions,Y_test_Positions, userInputData,timeSeriesData_BIG = create_The_Arrays_For_The_Models(userInputData_Original,timeSeriesData_BIG_Original,apply_functions_to_X_data = [],
                                     column_to_take =column_to_take,keepOpenDoorData =False,drop_other_positions =True ,closeDoorTraining_openDoorTest = False ,anyOtherMaskToBeUsed = None)

In [ ]:

columns_to_trim = abs(-30) + 15 -1
X_whole_true = np.empty((X_train.shape[0],X_train.shape[1]+X_test.shape[1],(X_train.shape[2] - columns_to_trim )))
for i,_ in enumerate(X_whole):
    data = np.vstack((X_train[i,:,:],X_test[i,:,:]))
    X_whole_true[i,:,:] = data[:,columns_to_trim:]
print(X_whole_true.shape)
X_whole_true

In [ ]:
for data in X_whole_true:
    negative_values = data[data < 0] 
    print(negative_values)

In [ ]:
min_value = {}
for i,data in enumerate(X_whole_true):
    min_value[i] =data[data > 0].min()
    print(min_value[i])

In [ ]:
min_value

In [ ]:
X_whole = np.empty((X_whole_true.shape))
for i,data in enumerate(X_whole_true):


    X_whole[i,:,:]  = np.where(data > 0, data, min_value[i])


In [ ]:
for data in X_whole:
    negative_values = data[data < 0] 
    print(negative_values)

In [ ]:
for i,data in enumerate(X_whole):

    data_flatten = data.ravel()
    
    plt.figure(figsize=(20, 3))  # ultra-wide & flat
    sns.boxplot(x=data_flatten)
    
    # Custom ticks for the first part of the plot
    ticks = np.arange(1, 3)  # 1,2,...,10
    #plt.xticks(ticks, labels=ticks)
    #plt.title(f"Box plot for sensor with id={i}")

    plt.show()

In [ ]:
for i,data in enumerate(X_whole):

    X_whole_flatten = data.ravel()
    
    plt.figure(figsize=(20, 3))  # ultra-wide & flat
    plt.title(f"Box plot with no outliers for sensor with id={i}")

    sns.boxplot(x=X_whole_flatten, showfliers=False)
    plt.show()

In [ ]:
for i,data in enumerate(X_whole):
    
    X_whole_flatten = data.ravel()
    
    plt.figure(figsize=(12, 4))
    sns.boxplot(x=X_whole_flatten)
    plt.xscale("log")
    # Custom ticks for the first part of the plot
    ticks = np.arange(1, 10)  # 1,2,...,10
   # plt.xticks(ticks, labels=ticks)
    plt.title(f"Box plot Log scaled for sensor with id={i}")
    plt.show()

In [ ]:
for sensor_id, data in enumerate(X_whole):
    X_whole_flatten = data.ravel()
    X_whole_flatten = np.sort(X_whole_flatten)
    
    n_plots = 4
    chunks = np.array_split(X_whole_flatten, n_plots)
    
    fig, axes = plt.subplots(n_plots, 1, figsize=(14, 10))
    
    # Figure-wide title
    fig.suptitle(f"Hist plots for sensor with id={sensor_id}", fontsize=16, y=1.02)
    
    for j, (chunk, ax) in enumerate(zip(chunks, axes)):
        sns.histplot(
            chunk,
            bins=50,
            ax=ax
        )
        ax.set_title(f"Histogram {j+1}")
    
    # Adjust layout to prevent overlap
    plt.tight_layout()
    plt.show()

## VOC ORIGINAL ROBUST SCALLER

### NO TRIMMING

In [ ]:
column_to_take ="VOC original"
X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,Y_train_Positions,Y_test_Positions, userInputData,timeSeriesData_BIG = create_The_Arrays_For_The_Models(userInputData_Original,timeSeriesData_BIG_Original,apply_functions_to_X_data = [],
                                     column_to_take =column_to_take,keepOpenDoorData =False,drop_other_positions =True ,closeDoorTraining_openDoorTest = False ,anyOtherMaskToBeUsed = None)

In [ ]:
X_train,X_test

In [ ]:
for i,data in enumerate(X_train):
    min_value =data[data > 0].min()
    print(data[data<0])
    X_train[i,:,:]  = np.where(data > 0, data, min_value)
    
for i,data in enumerate(X_test):
    min_value =data[data > 0].min()
    print(data[data<0])

    X_test[i,:,:]  = np.where(data > 0, data, min_value)
      


In [ ]:
X_whole = np.empty((X_train.shape[0],X_train.shape[1]+X_test.shape[1],X_train.shape[2]))
for i,_ in enumerate(X_whole):
    transformer = RobustScaler()
    X_train_data = transformer.fit_transform(X_train[i,:,:])
    X_test_data = transformer.transform(X_test[i,:,:])

    X_whole[i,:,:] = np.vstack((X_train_data,X_test_data))
print(X_whole.shape)
X_whole

In [ ]:
for i,data in enumerate(X_whole):

    data_flatten = data.ravel()
    
    plt.figure(figsize=(20, 3))  # ultra-wide & flat
    sns.boxplot(x=data_flatten)
    
    # Custom ticks for the first part of the plot
    ticks = np.arange(1, 3)  # 1,2,...,10
    #plt.xticks(ticks, labels=ticks)
    plt.title(f"Box plot for sensor with id={i}")

    plt.show()

In [ ]:
for i,data in enumerate(X_whole):

    X_whole_flatten = data.ravel()
    
    plt.figure(figsize=(20, 3))  # ultra-wide & flat
    plt.title(f"Box plot with no outliers for sensor with id={i}")

    sns.boxplot(x=X_whole_flatten, showfliers=False)
    plt.show()

In [ ]:
for i,data in enumerate(X_whole):
    
    X_whole_flatten = data.ravel()
    
    plt.figure(figsize=(12, 4))
    sns.boxplot(x=X_whole_flatten)
    plt.xscale("log")
    # Custom ticks for the first part of the plot
    ticks = np.arange(1, 10)  # 1,2,...,10
    #plt.xticks(ticks, labels=ticks)
    plt.title(f"Box plot Log scaled for sensor with id={i}")
    plt.show()

In [ ]:
for sensor_id, data in enumerate(X_whole):
    X_whole_flatten = data.ravel()
    X_whole_flatten = np.sort(X_whole_flatten)
    
    n_plots = 4
    chunks = np.array_split(X_whole_flatten, n_plots)
    
    fig, axes = plt.subplots(n_plots, 1, figsize=(14, 10))
    
    # Figure-wide title
    fig.suptitle(f"Hist plots for sensor with id={sensor_id}", fontsize=16, y=1.02)
    
    for j, (chunk, ax) in enumerate(zip(chunks, axes)):
        sns.histplot(
            chunk,
            bins=50,
            ax=ax
        )
        ax.set_title(f"Histogram {j+1}")
    
    # Adjust layout to prevent overlap
    plt.tight_layout()
    plt.show()

In [ ]:
userInputData_Original

In [ ]:
userInputData

## VOC ROBUST SCALLER

### NO TRIMMING

In [ ]:
column_to_take ="VOC"
X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,Y_train_Positions,Y_test_Positions, userInputData,timeSeriesData_BIG = create_The_Arrays_For_The_Models(userInputData_Original,timeSeriesData_BIG_Original,apply_functions_to_X_data = [],
                                     column_to_take =column_to_take,keepOpenDoorData =False,drop_other_positions =True ,closeDoorTraining_openDoorTest = False ,anyOtherMaskToBeUsed = None)

In [ ]:
X_train,X_test

In [ ]:
for i,data in enumerate(X_train):
    min_value =data[data > 0].min()
    print(data[data<0])
    X_train[i,:,:]  = np.where(data > 0, data, min_value)
    
for i,data in enumerate(X_test):
    min_value =data[data > 0].min()
    print(data[data<0])

    X_test[i,:,:]  = np.where(data > 0, data, min_value)
      


In [ ]:
X_whole = np.empty((X_train.shape[0],X_train.shape[1]+X_test.shape[1],X_train.shape[2]))
for i,_ in enumerate(X_whole_true):
    transformer = RobustScaler()

    X_train_data = transformer.fit_transform(X_train[i,:,:])
    X_test_data = transformer.transform(X_test[i,:,:])

    X_whole[i,:,:] = np.vstack((X_train_data,X_test_data))
print(X_whole.shape)
X_whole

In [ ]:
for i,data in enumerate(X_whole):

    data_flatten = data.ravel()
    
    plt.figure(figsize=(20, 3))  # ultra-wide & flat
    sns.boxplot(x=data_flatten)
    
    # Custom ticks for the first part of the plot
    ticks = np.arange(1, 3)  # 1,2,...,10
    #plt.xticks(ticks, labels=ticks)
    #plt.title(f"Box plot for sensor with id={i}")

    plt.show()

In [ ]:
for i,data in enumerate(X_whole):

    X_whole_flatten = data.ravel()
    
    plt.figure(figsize=(20, 3))  # ultra-wide & flat
    plt.title(f"Box plot with no outliers for sensor with id={i}")

    sns.boxplot(x=X_whole_flatten, showfliers=False)
    plt.show()

In [ ]:
for i,data in enumerate(X_whole):
    
    X_whole_flatten = data.ravel()
    
    plt.figure(figsize=(12, 4))
    sns.boxplot(x=X_whole_flatten)
    plt.xscale("log")
    # Custom ticks for the first part of the plot
    ticks = np.arange(1, 10)  # 1,2,...,10
    #plt.xticks(ticks, labels=ticks)
    plt.title(f"Box plot Log scaled for sensor with id={i}")
    plt.show()

In [ ]:
for sensor_id, data in enumerate(X_whole):
    X_whole_flatten = data.ravel()
    X_whole_flatten = np.sort(X_whole_flatten)
    
    n_plots = 4
    chunks = np.array_split(X_whole_flatten, n_plots)
    
    fig, axes = plt.subplots(n_plots, 1, figsize=(14, 10))
    
    # Figure-wide title
    fig.suptitle(f"Hist plots for sensor with id={sensor_id}", fontsize=16, y=1.02)
    
    for j, (chunk, ax) in enumerate(zip(chunks, axes)):
        sns.histplot(
            chunk,
            bins=50,
            ax=ax
        )
        ax.set_title(f"Histogram {j+1}")
    
    # Adjust layout to prevent overlap
    plt.tight_layout()
    plt.show()

## VOC ORIGINAL STAMDARD SCALLER

### NO TRIMMING

In [ ]:
column_to_take ="VOC original"
X_train,X_test,Y_train_Euclidians,Y_test_Euclidians,Y_train_Positions,Y_test_Positions, userInputData,timeSeriesData_BIG = create_The_Arrays_For_The_Models(userInputData_Original,timeSeriesData_BIG_Original,apply_functions_to_X_data = [],
                                     column_to_take =column_to_take,keepOpenDoorData =False,drop_other_positions =True ,closeDoorTraining_openDoorTest = False ,anyOtherMaskToBeUsed = None)

In [ ]:
X_train,X_test

In [ ]:
for i,data in enumerate(X_train):
    min_value =data[data > 0].min()
    print(data[data<0])
    X_train[i,:,:]  = np.where(data > 0, data, min_value)
    
for i,data in enumerate(X_test):
    min_value =data[data > 0].min()
    print(data[data<0])

    X_test[i,:,:]  = np.where(data > 0, data, min_value)
      


In [ ]:
X_whole = np.empty((X_train.shape[0],X_train.shape[1]+X_test.shape[1],X_train.shape[2]))
for i,_ in enumerate(X_whole):
    transformer = StandardScaler()
    
    X_train_data = transformer.fit_transform(X_train[i,:,:])
    X_test_data = transformer.transform(X_test[i,:,:])

    X_whole[i,:,:] = np.vstack((X_train_data,X_test_data))
print(X_whole.shape)
X_whole

In [ ]:
for i,data in enumerate(X_whole):

    data_flatten = data.ravel()
    
    plt.figure(figsize=(20, 3))  # ultra-wide & flat
    sns.boxplot(x=data_flatten)
    
    # Custom ticks for the first part of the plot
    ticks = np.arange(1, 3)  # 1,2,...,10
    #plt.xticks(ticks, labels=ticks)
    plt.title(f"Box plot for sensor with id={i}")

    plt.show()

In [ ]:
for i,data in enumerate(X_whole):

    X_whole_flatten = data.ravel()
    
    plt.figure(figsize=(20, 3))  # ultra-wide & flat
    plt.title(f"Box plot with no outliers for sensor with id={i}")

    sns.boxplot(x=X_whole_flatten, showfliers=False)
    plt.show()

In [ ]:
for i,data in enumerate(X_whole):
    
    X_whole_flatten = data.ravel()
    
    plt.figure(figsize=(12, 4))
    sns.boxplot(x=X_whole_flatten)
    plt.xscale("log")
    # Custom ticks for the first part of the plot
    ticks = np.arange(1, 10)  # 1,2,...,10
    #plt.xticks(ticks, labels=ticks)
    plt.title(f"Box plot Log scaled for sensor with id={i}")
    plt.show()

In [ ]:
for sensor_id, data in enumerate(X_whole):
    X_whole_flatten = data.ravel()
    X_whole_flatten = np.sort(X_whole_flatten)
    
    n_plots = 4
    chunks = np.array_split(X_whole_flatten, n_plots)
    
    fig, axes = plt.subplots(n_plots, 1, figsize=(14, 10))
    
    # Figure-wide title
    fig.suptitle(f"Hist plots for sensor with id={sensor_id}", fontsize=16, y=1.02)
    
    for j, (chunk, ax) in enumerate(zip(chunks, axes)):
        sns.histplot(
            chunk,
            bins=50,
            ax=ax
        )
        ax.set_title(f"Histogram {j+1}")
    
    # Adjust layout to prevent overlap
    plt.tight_layout()
    plt.show()